In [1]:
import pandas as pd
import numpy as np

# 1. Khởi tạo lại bảng Geography chuẩn (như bài trước) để lấy Khóa (Geography_ID)
df_geo_raw = pd.read_csv('geography.csv')
df_geo_raw = df_geo_raw.rename(columns={'zip': 'Postal_Code', 'city': 'City', 'region': 'Region', 'district': 'State'})
df_geo_raw['Country'] = 'Vietnam'
df_geo_raw = df_geo_raw.sort_values(by=['City', 'Postal_Code']).reset_index(drop=True)
df_geo_raw = df_geo_raw.drop_duplicates(subset=['Postal_Code', 'City', 'State', 'Country'])
df_geo_raw.insert(0, 'Geography_ID', range(1, len(df_geo_raw) + 1))

# 2. Đọc file customers.csv
df_cust = pd.read_csv('customers.csv')
print(f"Tổng số khách hàng ban đầu: {len(df_cust)}")

# 3. Kết nối (Merge) để đối chiếu mã 'zip' và lấy 'Geography_ID'
df_merged = df_cust.merge(
    df_geo_raw[['Postal_Code', 'Geography_ID']], 
    left_on='zip', 
    right_on='Postal_Code', 
    how='left'
)

# Kiểm tra xem có khách hàng nào không lấy được Geography_ID không (Do lỗi zip)
missing_geo = df_merged['Geography_ID'].isnull().sum()
print(f"Số khách hàng không ánh xạ được khu vực: {missing_geo}")

Tổng số khách hàng ban đầu: 121930
Số khách hàng không ánh xạ được khu vực: 0


In [2]:
# 1. Trích xuất các cột cần thiết
df_customer_final = df_merged[['customer_id', 'Geography_ID']].copy()

# Đổi tên cho đúng chuẩn thiết kế
df_customer_final.rename(columns={'customer_id': 'Customer_ID'}, inplace=True)

# 2. Xử lý Dữ liệu Email (Tự động sinh dựa trên Customer_ID)
# Ví dụ: customer1@example.com
df_customer_final['Email'] = 'customer' + df_customer_final['Customer_ID'].astype(str) + '@example.com'

# 3. Xử lý Dữ liệu Số điện thoại (Tạo số random định dạng VN)
np.random.seed(42)  # Cố định seed để random ra kết quả giống nhau ở mọi lần chạy
phones = ["+84" + str(np.random.randint(900000000, 999999999)) for _ in range(len(df_customer_final))]
df_customer_final['Phone'] = phones

# 4. Sắp xếp lại thứ tự cột cho đúng format yêu cầu
final_columns = ['Customer_ID', 'Email', 'Phone', 'Geography_ID']
df_customer_final = df_customer_final[final_columns]

print("\nDữ liệu mẫu sau khi hoàn thiện:")
display(df_customer_final.head())


Dữ liệu mẫu sau khi hoàn thiện:


,Customer_ID,Email,Phone,Geography_ID
0,1,customer1@example.com,+84965682867,10968
1,2,customer2@example.com,+84956755036,10968
2,3,customer3@example.com,+84956882282,10968
3,4,customer4@example.com,+84921081788,10968
4,5,customer5@example.com,+84913315092,10968


In [4]:
# 1. Kiểm tra Dữ liệu Missing/Null
print("Kiểm tra Null:")
print(df_customer_final.isnull().sum())

# 2. Kiểm tra Dữ liệu Trùng lặp (Unique Processing)
# Đảm bảo mỗi Customer_ID là độc nhất
duplicates_count = df_customer_final.duplicated(subset=['Customer_ID']).sum()
print(f"\nSố lượng dòng trùng lặp: {duplicates_count}")

# (Tùy chọn) Xóa trùng lặp nếu có
if duplicates_count > 0:
    df_customer_final = df_customer_final.drop_duplicates(subset=['Customer_ID'])
    print("-> Đã xóa dữ liệu trùng lặp.")

# 3. Lưu ra file chuẩn hóa
output_file = 'customers_silver.csv'
df_customer_final.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n Đã xử lý xong và lưu {len(df_customer_final)} bản ghi vào file {output_file}!")

Kiểm tra Null:
Customer_ID     0
Email           0
Phone           0
Geography_ID    0
dtype: int64

Số lượng dòng trùng lặp: 0

 Đã xử lý xong và lưu 121930 bản ghi vào file customers_silver.csv!
